In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from collections import Counter
import os
import pandas as pd

In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(path)

In [ ]:
# Task 2: Write your code here:
df.head(-5)

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
missing_percentage = (df.isnull().sum() / len(df)) * 100
missing_data = pd.DataFrame({
    'Column': missing_percentage.index,
    'Missing_Percentage': missing_percentage.values
})
missing_data = missing_data[missing_data['Missing_Percentage'] > 0].sort_values('Missing_Percentage', ascending=False)

print("Missing Data Analysis:")
missing_data.head(10)

In [ ]:
df_clean = df.copy()

In [ ]:
# Task 1: Write your code here:
# Fill categorical columns with 'unknown' - missing likely means "not specified"
for col in df_clean.columns:
    df_clean[col] = df_clean[col].fillna('unknown')


print("Missing values remaining:", df_clean.isnull().sum().sum())

In [ ]:
# Task 2: Write your code here:

def check_duplicates(df):

  #TODO: get duplicated data using pandas
  duplicates = df_clean.duplicated().sum()

  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    df_clean.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(df_clean)

In [ ]:
df_clean.info()

In [ ]:
#categorical columns
categorical_cols = df_clean.select_dtypes(include=["object"]).columns

print("Categorical Columns:", list(categorical_cols))

In [ ]:
# Task 3: Write your code here:
from sklearn.preprocessing import LabelEncoder

label_encoders = {}
for col in categorical_cols:
  le = LabelEncoder()
  df_clean[col] = le.fit_transform(df[col])
  label_encoders[col] = le

df_clean

In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

numerical_cols = df_clean.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET

scaler = StandardScaler()
df_clean[numerical_cols] = scaler.fit_transform(df_clean[numerical_cols])
df_clean.head()


In [ ]:
import seaborn as sns

In [ ]:
# Task 5: Write your code here:
def check_target_imbalance(df, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df_clean, "Target")

In [ ]:
#target is imbalanced majority is 0

In [ ]:
# Task 1: Write your code here:
X = df_clean.drop(['Target'], axis = 1)
y = df_clean['Target']

In [ ]:
%pip install catboost

In [ ]:
# Task 2,3,4,5: Write your code here:

In [ ]:
from catboost import CatBoostClassifier

In [ ]:
model = CatBoostClassifier(verbose=0, n_estimators=320, max_depth=4)

In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

In [ ]:
n_splits = 5 # K

# Stratified 5-Fold Cross-Validation, shuffled
skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

In [ ]:
# Storage for logistic regression results for each fold

accuracy = []
precision = []
recall = []
f1 = []

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    # Calculate evaluation metrics
    a = accuracy_score(y_test, y_pred)
    p = precision_score(y_test, y_pred, zero_division=0)
    r = recall_score(y_test, y_pred, zero_division=0)
    f = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    accuracy.append(a)
    precision.append(p)
    recall.append(r)
    f1.append(f)

print(f"  Accuracy:  {np.mean(accuracy):.4f}")
print(f"  Precision: {np.mean(precision):.4f}")
print(f"  Recall:    {np.mean(recall):.4f}")
print(f"  F1-Score:  {np.mean(f1):.4f}")

In [ ]:
importances = model.feature_importances_
plt.figure(figsize=(10,30))
plt.barh(X.columns, importances)
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
#plt.yticks(rotation=45)
plt.show()

In [ ]:
# Task 2: Write your code here:
df_clean['P_2']

In [ ]:
# Task Bonus: Write your code here:
Xp = df_clean['P_2']


In [ ]:
Xp

In [ ]:
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
    print(f"\nFold {fold_idx + 1}/{n_splits}")

    Xp_train, Xp_test = Xp.iloc[train_index], Xp.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(Xp_train, y_train)
    y_pred = model.predict(Xp_test)


    # Calculate evaluation metrics
    a = accuracy_score(y_test, y_pred)
    p = precision_score(y_test, y_pred, zero_division=0)
    r = recall_score(y_test, y_pred, zero_division=0)
    f = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    accuracy.append(a)
    precision.append(p)
    recall.append(r)
    f1.append(f)

print(f"  Accuracy:  {np.mean(accuracy):.4f}")
print(f"  Precision: {np.mean(precision):.4f}")
print(f"  Recall:    {np.mean(recall):.4f}")
print(f"  F1-Score:  {np.mean(f1):.4f}")